A framework designed to compare performance of retrieved results based on word-level embeddings, sentence-level embeddings, and embeddings pre-trained along with graph metrics.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel
from sentence_transformers import SentenceTransformer
import csv
import ast
from sklearn.metrics.pairwise import cosine_similarity
import ollama

1. Vector Database Organization

In [2]:
INIT_WORD_VECTOR_DB = [] #Dataset found here: https://huggingface.co/ngxson/demo_simple_rag_py/blob/main/cat-facts.txt

In [3]:
with open('./cat-facts.txt', encoding='UTF-8') as file:
    INIT_WORD_VECTOR_DB = file.readlines()

1.1. BERT Embedding Based DB

In [ ]:
BERT_TOKENIZER = BertTokenizer.from_pretrained('bert-base-uncased')
BERT_MODEL = BertModel.from_pretrained('bert-base-uncased')

In [5]:
BERT_VECTOR_DB =  []

In [7]:
def bertSentToEmbedding(sent: str) -> Tensor:

    tokenizedLine = BERT_TOKENIZER.tokenize(sent)
    
    inputs = torch.tensor(BERT_TOKENIZER.convert_tokens_to_ids(tokenizedLine)).unsqueeze(0)
    
    with torch.no_grad():
    
        output = BERT_MODEL(inputs)
    
    embeddings = F.normalize(output.last_hidden_state[:, 0, :], p = 2, dim = 1)

    return embeddings

In [107]:
#Creating vector database based on BERT-model derived sentence-level embeddings.

for line in INIT_WORD_VECTOR_DB:

    embedding = bertSentToEmbedding(line)

    BERT_VECTOR_DB.append(embedding)

In [110]:
with open("BERT_VECTOR_DB_1.csv", 'w',  encoding = 'utf-8') as file:
    
    writer  = csv.writer(file)

    writer.writerow(["Sentences", "BERT Vectors"])

    for i in range(len(BERT_VECTOR_DB)):
        
        writer.writerow([INIT_WORD_VECTOR_DB[i], BERT_VECTOR_DB[i]])
        
        

In [6]:
def cleanStoredBertTensor(val: str):
    
    startingIndex = 0
    endingIndex = 0
    
    endingFound = False

    for i in range(len(val)):

        if val[i] == '[':
            startingIndex = i

    for i in range(len(val)):

        if val[i] == ']' and not endingFound:
            endingIndex = i
            endingFound = True

    return ast.literal_eval(val[startingIndex:endingIndex + 1])
    


In [38]:
BERT_VECTOR_DB = pd.read_csv("BERT_VECTOR_DB_1.csv")

In [39]:
BERT_VECTOR_DB 

,Sentences,BERT Vectors
0,"On average, cats spend 2/3 of every day sleepi...","tensor([[ 3.2025e-02, -1.9272e-03, 7.5102e-03..."
1,"Unlike dogs, cats do not have a sweet tooth. S...","tensor([[ 2.5943e-02, 1.3421e-02, -1.0437e-03..."
2,"When a cat chases its prey, it keeps its head ...","tensor([[ 4.9998e-02, 7.0072e-03, 1.5407e-02..."
3,The technical term for a cat’s hairball is a “...,"tensor([[ 2.2468e-03, 6.4672e-03, 2.8188e-02..."
4,A group of cats is called a “clowder.”\r\n,"tensor([[-9.7049e-04, 4.5340e-03, 3.1755e-02..."
...,...,...
145,"Like birds, cats have a homing ability that us...","tensor([[ 3.0573e-02, 1.5667e-02, 2.0914e-02..."
146,Cats bury their feces to cover their trails fr...,"tensor([[ 1.8749e-02, -9.4188e-03, 1.5741e-02..."
147,Cats sleep 16 to 18 hours per day. When cats a...,"tensor([[ 2.3224e-02, 1.4838e-02, 2.2194e-02..."
148,"Besides smelling with their nose, cats can sme...","tensor([[ 1.1026e-02, 5.3487e-02, 1.8455e-02..."


In [57]:
req = input("Please input your request about cats.")

reqEmbedding = bertSentToEmbedding(req)

In [58]:
bertGreatestMatch = None 
bertGreatestCosSim = -10

for row in BERT_VECTOR_DB.iterrows():
    
    dbVector = cleanStoredBertTensor(row[1]['BERT Vectors'])
    
    dbVector = torch.tensor(dbVector).reshape(1, -1)
    
    dbReqCos = cosine_similarity(dbVector, reqEmbedding)[0][0]

    if dbReqCos > bertGreatestCosSim:
        
        bertGreatestMatch = row[1]['Sentences']
    
        bertGreatestCosSim = dbReqCos   

In [59]:
print("Query: {}\nResponse: {}".format(req, bertGreatestMatch))

Query: When was the first cat show?
Response: Researchers believe the word “tabby” comes from Attabiyah, a neighborhood in Baghdad, Iraq. Tabbies got their name because their striped coats resembled the famous wavy patterns in the silk produced in this city.



2. S-BERT Embedding Based DB

In [ ]:
SBERT = SentenceTransformer("all-MiniLM-L6-v2")

In [61]:
SBERT_VECTOR_DB = []

In [17]:
def sbertSentToEmbedding(sent: str) -> Tensor:

    embeddings = SBERT.encode(sent, normalize_embeddings = True)

    return embeddings

In [ ]:
for line in INIT_WORD_VECTOR_DB:

    embedding = sbertSentToEmbedding(line)

    SBERT_VECTOR_DB.append(embedding)
    

In [196]:
with open("SBERT_VECTOR_DB_1.csv", 'w',  encoding = 'utf-8') as file:
    
    writer  = csv.writer(file)

    writer.writerow(["Sentences", "SBERT Vectors"])

    for i in range(len(BERT_VECTOR_DB)):
        
        writer.writerow([INIT_WORD_VECTOR_DB[i], SBERT_VECTOR_DB[i]])
        

In [62]:
SBERT_VECTOR_DB = pd.read_csv("SBERT_VECTOR_DB_1.csv")

In [18]:
def cleanStoredSbertTensor(val: str):
    
    start = val.find('[')
    end = val.rfind(']')
    
    if start == -1 or end == -1:
        return "" # Handle empty or corrupt strings safely
        
    return val[start:end + 1]

In [63]:
req = input("Please input your request about cats.")

reqEmbedding = sbertSentToEmbedding(req)

In [66]:
sbertGreatestMatch = None 
sbertGreatestCosSim = -10

for row in SBERT_VECTOR_DB.iterrows():
    
    dbVector = cleanStoredSbertTensor(row[1]['SBERT Vectors'])
    
    numericalVector = [float(x) for x in dbVector.strip("[]").split()]

    numericalVector = torch.tensor(numericalVector).reshape(1, -1)
        
    dbReqCos = cosine_similarity(numericalVector, reqEmbedding.reshape(1, -1))
    
    if dbReqCos > sbertGreatestCosSim:
            
        sbertGreatestMatch = row[1]['Sentences']
        sbertGreatestCosSim = dbReqCos

In [67]:
print("Query: {}\nResponse: {}".format(req, sbertGreatestMatch))

Query: When was the first cat show?
Response: The first cat show was organized in 1871 in London. Cat shows later became a worldwide craze.



2.1. Augmentation and Generation

2.1.1. Decoder Model 

In [68]:
LLAMA_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

2.1.2. Augmentation

2.1.2.1. BERT-Informed Augmentation

In [71]:
#Augmentation based on BERT-selected most similar vector.

bertInstructionPrompt = "Acting as a helpful chat, please answer the question asked using only the following pieces of context. Please do not make up any new information. Retrieved context: {}".format(bertGreatestMatch)

In [72]:
print(bertInstructionPrompt)

Acting as a helpful chat, please answer the question asked using only the following pieces of context. Please do not make up any new information. Retrieved context: Researchers believe the word “tabby” comes from Attabiyah, a neighborhood in Baghdad, Iraq. Tabbies got their name because their striped coats resembled the famous wavy patterns in the silk produced in this city.



2.1.2.2. S-BERT-Informed Augmentation

In [74]:
#Augmentation based on S-BERT-selected most similar vector.

sbertInstructionPrompt = "Acting as a helpful chat, please answer the question asked using only the following pieces of context. Please do not make up any new information. Retrieved context: {}".format(sbertGreatestMatch)

In [75]:
print(sbertInstructionPrompt)

Acting as a helpful chat, please answer the question asked using only the following pieces of context. Please do not make up any new information. Retrieved context: The first cat show was organized in 1871 in London. Cat shows later became a worldwide craze.



2.1.3. Generation

2.1.3.1. BERT-Informed Generation

In [76]:
#Sample generation tested with BERT-based vector database retrieval.

bertRefLLAMAStream = ollama.chat(
    model = LLAMA_MODEL,
    messages = [
        {'role': 'system', 'content': bertInstructionPrompt},
        {'role': 'user', 'content': req},
    ]
)

In [77]:
print("(BERT-Informed Retrieval)\nQuery: {}\nChatbot Response: {}".format(req, bertRefLLAMAStream.message.content))

(BERT-Informed Retrieval)
Query: When was the first cat show?
Chatbot Response: There is no record of a "cat show" being held in Baghdad, Iraq, or at Attabiyah, as these places do not have a significant history of cat shows.


2.1.3.2. S-BERT-Informed Generation

In [78]:
#Sample generation tested with S-BERT-based vector database retrieval.

sbertRefLLAMAStream = ollama.chat(
    model = LLAMA_MODEL,
    messages = [
        {'role': 'system', 'content': sbertInstructionPrompt},
        {'role': 'user', 'content': req},
    ]
)

In [79]:
print("(S-BERT-Informed Retrieval)\nQuery: {}\nChatbot Response: {}".format(req, sbertRefLLAMAStream.message.content))

(S-BERT-Informed Retrieval)
Query: When was the first cat show?
Chatbot Response: The first cat show was organized in 1871 in London.
